# Convert SmartSPIM cell detection to LightSuite Sample Space v1

This notebook converts **SmartSPIM / LCT cell-detection** exports into formats accepted by
`lightsuite brain import-annotations`:

| LightSuite format | SmartSPIM source | Use when |
|-------------------|------------------|----------|
| `points_csv` | `*_points_*.json` (`[[z,y,x],…]` 0-based) | Cell positions (preferred) |
| `mask_tiff` | `*_cells.zarr` / `level_01` (optional) | Full binary mask — large I/O |

**Do not import** `*_projected_points.json` or `*_points_in_region.csv` — those are already
atlas-mapped / region-aggregated by the detection software. LightSuite re-warps native
coordinates with *your* registration transform.

**Before you start:**

1. Split flat `All_Channels` into `Ch0`/`Ch1`/`Ch2` if needed:
   `uv run lightsuite brain split-smartspim-channels -s /path/to/All_Channels`
2. Run `lightsuite brain preprocess` so `sample_reference.json` exists.

CLI alternative for points only:

```bash
uv run lightsuite brain convert-smartspim-points \
  -s /path/to/488_points_Endogenous.json \
  -o /path/to/converted/smartspim_488_points.csv
```

See also: [Annotation import](../../docs/annotation_import.md).

## 0. Configure paths

Edit the variables below for your sample. Defaults point at the Jules SmartSPIM dataset
on the lab share (adjust if paths differ).

In [ ]:
from pathlib import Path

# Where LightSuite wrote / will write preprocess outputs (sample_reference.json)
SAVE_PATH = Path("/media/gbm/NVME2/ALICe-pipelines-data/Jules/registration_allen")

_jules = (
    "/run/user/1002/gvfs/smb-share:server=h1data1.wysscenter.ch,"
    "share=computingdata/Alice/0003_CBT_WYSS_LIGHTSHEET/DATA/SMARTSPIM/Jules"
)
JULES = Path("".join(_jules))
ALL_CHANNELS = JULES / "All_Channels"
DETECTION_DIR = JULES / "cell_detection"
POINTS_JSON = DETECTION_DIR / "488_points_Endogenous.json"
CELLS_ZARR = DETECTION_DIR / "488_Endogenous_cells.zarr"  # optional mask source

OUTPUT_DIR = Path("/media/gbm/NVME2/ALICe-pipelines-data/Jules/segmentation/converted")
LABEL = "smartspim_488"  # used in output filenames and YAML import.label

WRITE_MASK = False  # set True only if you need mask_tiff (streams full-res zarr)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("save_path:", SAVE_PATH)
print("Jules root:", JULES)
print("All_Channels:", ALL_CHANNELS)
print("points JSON:", POINTS_JSON)
print("cells zarr:", CELLS_ZARR)
print("output:", OUTPUT_DIR)


## 1. (Optional) Split flat `All_Channels` → `Ch0` / `Ch1` / `Ch2`

Jules-style SmartSPIM writes interleaved planes in one folder
(`Z000000_Ch0.tif`, `Z000000_Ch1.tif`, …). LightSuite `planeperfile` multi-channel
configs need one directory per channel.

Default mode is **symlink** (reversible, no extra disk). Use `mode="move"` to relocate
files into the `ChN` folders.

In [ ]:
from lightsuite.io.smartspim_channels import SplitMode, split_smartspim_all_channels

RUN_SPLIT = True  # set False if Ch0/Ch1 already exist

if RUN_SPLIT:
    if not ALL_CHANNELS.is_dir():
        raise FileNotFoundError(f"Missing {ALL_CHANNELS}")
    # Skip if already split
    if (ALL_CHANNELS / "Ch0").is_dir() and any((ALL_CHANNELS / "Ch0").glob("*.tif*")):
        print("Ch0 already populated — skipping split")
        result = None
    else:
        result = split_smartspim_all_channels(
            ALL_CHANNELS,
            mode=SplitMode.SYMLINK,
            max_channels=3,
        )
        for ch, path in sorted(result.channel_dirs.items()):
            print(f"Ch{ch}: {result.plane_counts[ch]} planes → {path}")
else:
    result = None
    print("RUN_SPLIT=False")

print("\nYAML snippet for preprocess:")
ch_dirs = sorted((ALL_CHANNELS / f"Ch{i}" for i in range(3) if (ALL_CHANNELS / f"Ch{i}").is_dir()))
for i, d in enumerate(ch_dirs):
    prefix = "channels:\n    - " if i == 0 else "    - "
    print(f"{prefix}{d}")

## 2. Load the LightSuite native grid contract

After `preprocess`, LightSuite writes `sample_reference.json`. Every imported annotation must
align with this grid:

- **Shape** `(Y, X, Z)` = `shape_yxz` → `(ny, nx, nz)`
- **Voxel size** `[x, y, z]` in µm
- **Indices** are **1-based** — the first voxel center is `(1, 1, 1)`
- **Axis order** for point CSVs is always `x, y, z`

If preprocess has not been run yet, set `REFERENCE_OVERRIDE` from acquisition metadata
(Jules: `(8802, 7412, 1571)`, `voxel_um=[1.8, 1.8, 4.0]`).

In [ ]:
import json

REFERENCE_OVERRIDE = None
# REFERENCE_OVERRIDE = {
#     "format": "lightsuite_sample_space_v1",
#     "sample_name": "Jules",
#     "shape_yxz": [8802, 7412, 1571],
#     "voxel_um": [1.8, 1.8, 4.0],
#     "index_base": 1,
#     "axis_order": "xyz",
#     "coordinate_units": "voxel_indices",
#     "orientation_applied": False,
# }

ref_path = SAVE_PATH / "sample_reference.json"
if REFERENCE_OVERRIDE is not None:
    reference = REFERENCE_OVERRIDE
    print("Using REFERENCE_OVERRIDE")
elif ref_path.is_file():
    reference = json.loads(ref_path.read_text(encoding="utf-8"))
else:
    raise FileNotFoundError(
        f"Missing {ref_path}. Run preprocess, or set REFERENCE_OVERRIDE above.\n"
        "  uv run lightsuite brain preprocess -c your_config.yaml"
    )

ny, nx, nz = reference["shape_yxz"]
voxel_um = reference["voxel_um"]
print(json.dumps(reference, indent=2))
print()
print(f"Expected mask shape (Y, X, Z): ({ny}, {nx}, {nz})")
print(f"Point bounds: 1 <= x <= {nx}, 1 <= y <= {ny}, 1 <= z <= {nz}")

## 3. Convert detection JSON → `points.csv`

SmartSPIM detection stores **0-based** voxel indices as **`[z, y, x]`** on the same grid as
`All_Channels`. LightSuite needs **1-based** **`x, y, z`**:

```python
x = zyx[2] + 1
y = zyx[1] + 1
z = zyx[0] + 1
```

In [ ]:
import numpy as np

from lightsuite.import_.smartspim_detection import convert_smartspim_points_json_to_csv

POINTS_CSV = OUTPUT_DIR / f"{LABEL}_points.csv"
n = convert_smartspim_points_json_to_csv(POINTS_JSON, POINTS_CSV)
points_xyz = np.loadtxt(POINTS_CSV, delimiter=",", skiprows=1)
if points_xyz.ndim == 1:
    points_xyz = points_xyz.reshape(1, -1)

print(f"Wrote {n} points → {POINTS_CSV}")
print(
    "Ranges (1-based):"
    f" x [{points_xyz[:, 0].min():.1f}, {points_xyz[:, 0].max():.1f}]"
    f" y [{points_xyz[:, 1].min():.1f}, {points_xyz[:, 1].max():.1f}]"
    f" z [{points_xyz[:, 2].min():.1f}, {points_xyz[:, 2].max():.1f}]"
)

## 4. Validate points against `sample_reference.json`

In [ ]:
if len(points_xyz) == 0:
    raise RuntimeError("No points written")

inside = (
    (points_xyz[:, 0] >= 1)
    & (points_xyz[:, 0] <= nx)
    & (points_xyz[:, 1] >= 1)
    & (points_xyz[:, 1] <= ny)
    & (points_xyz[:, 2] >= 1)
    & (points_xyz[:, 2] <= nz)
)
n_out = int((~inside).sum())
print(f"In bounds: {int(inside.sum())}/{len(points_xyz)} ({100 * inside.mean():.2f}%)")
if n_out:
    print(f"WARNING: {n_out} points outside native grid — check axis order / grid size")
else:
    print("All points within sample_reference bounds")

## 5. Optional — stream zarr mask → `mask.tif`

Full-resolution `level_01` is a binary `(Z, Y, X)` volume matching `All_Channels`.
Writing a multipage TIFF is optional and **expensive** over SMB; prefer `points_csv`.

Set `WRITE_MASK = True` in section 0 to enable.

In [ ]:
MASK_TIFF = OUTPUT_DIR / f"{LABEL}_mask.tif"

if not WRITE_MASK:
    print("WRITE_MASK=False — skipping zarr → mask.tif")
else:
    import zarr
    import tifffile

    g = zarr.open_group(str(CELLS_ZARR), mode="r")
    level = g["level_01"]
    z_shape, y_shape, x_shape = (int(v) for v in level.shape)
    print(f"zarr level_01 shape (Z,Y,X): {(z_shape, y_shape, x_shape)}")
    if (y_shape, x_shape, z_shape) != (ny, nx, nz):
        raise ValueError(
            f"Mask shape {(y_shape, x_shape, z_shape)} != sample_reference {(ny, nx, nz)}"
        )

    with tifffile.TiffWriter(MASK_TIFF, bigtiff=True) as writer:
        for zi in range(z_shape):
            plane = np.asarray(level[zi], dtype=np.uint8)
            plane = (plane > 0).astype(np.uint8) * 255
            writer.write(plane, photometric="minisblack")
            if zi % 100 == 0 or zi == z_shape - 1:
                print(f"  wrote Z {zi + 1}/{z_shape}")
    print(f"Wrote mask → {MASK_TIFF}")

## 6. YAML snippet for `import-annotations`

In [ ]:
import yaml

ann = [
    {
        "format": "points_csv",
        "path": str(POINTS_CSV),
        "label": LABEL,
    }
]
if WRITE_MASK and MASK_TIFF.is_file():
    ann.append(
        {
            "format": "mask_tiff",
            "path": str(MASK_TIFF),
            "label": f"{LABEL}_mask",
        }
    )

snippet = {"import": {"write_csv": True, "annotations": ann}}
print(yaml.safe_dump(snippet, sort_keys=False))
print("# After register:")
print("#   uv run lightsuite brain import-annotations -c your_config.yaml")

## 7. Test load with LightSuite adapters (optional)

In [ ]:
from lightsuite.config.models import AnnotationFormat, AnnotationImportConfig
from lightsuite.import_.adapters import load_points_csv, prepare_points_for_sample
from lightsuite.import_.sample_reference import SampleReference, load_sample_reference

if ref_path.is_file():
    ref = load_sample_reference(SAVE_PATH)
else:
    ref = SampleReference(
        format="lightsuite_sample_space_v1",
        sample_name=str(reference.get("sample_name", "sample")),
        shape_yxz=[int(ny), int(nx), int(nz)],
        voxel_um=[float(v) for v in voxel_um],
        index_base=1,
        axis_order="xyz",
        coordinate_units="voxel_indices",
        orientation_applied=False,
    )

spec = AnnotationImportConfig(
    format=AnnotationFormat.POINTS_CSV,
    path=POINTS_CSV,
    label=LABEL,
)
loaded = load_points_csv(spec)
prepared = prepare_points_for_sample(loaded, reference=ref)
print(
    f"Points: {loaded.coordinates.shape[0]} loaded → "
    f"{prepared.coordinates.shape[0]} in bounds"
)
